In [1]:
# AI-Powered Legal Document Review System as Part of Capstone Project for
# Developer Friendly Track- Microsoft Agentic AI Agentic Engineer (Batch 1: 9th to 25th march, 2026)
# Sponsored by Accenture, Author- Tithi Biswas
#
# This notebook builds a legal document review pipeline with:
# 1. Document ingestion to Azure Blob Storage
# 2. Vector indexing in Azure AI Search
# 3. Clause extraction with Azure OpenAI
# 4. RAG-based compliance validation
# 5. Grounded summary generation
# 6. Basic testing examples

In [2]:
# Install dependencies
%pip install -q openai python-dotenv pydantic pypdf tenacity azure-storage-blob azure-search-documents azure-ai-projects

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 1) Setup and configuration
import os
import json
import re
from pathlib import Path
from typing import Any, List, Dict
from dotenv import load_dotenv
from pypdf import PdfReader
from openai import OpenAI
from azure.core.credentials import AzureKeyCredential
from azure.storage.blob import BlobServiceClient
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SearchableField,
    SimpleField,
    VectorSearch,
    VectorSearchProfile,
    HnswAlgorithmConfiguration,
)
from azure.search.documents.models import VectorizedQuery

load_dotenv()

# ========= USER SETTINGS =========
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "key")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "endpoint")
AZURE_OPENAI_MODEL = os.getenv("AZURE_OPENAI_MODEL", "gpt-4o")
AZURE_OPENAI_EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-ada-002")

# Kept for compatibility
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-11-20")

AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT", "endpoint")
AZURE_SEARCH_KEY = os.getenv("AZURE_SEARCH_KEY", "key")
AZURE_SEARCH_INDEX_NAME = os.getenv("AZURE_SEARCH_INDEX_NAME", "legal_doc_review")

AZURE_STORAGE_CONNECTION_STRING = os.getenv("AZURE_STORAGE_CONNECTION_STRING", "connection string")
AZURE_STORAGE_CONTAINER = os.getenv("AZURE_STORAGE_CONTAINER", "legal-doc-review")

MAX_CHUNK_CHARS = int(os.getenv("MAX_CHUNK_CHARS", "4000"))
CHUNK_OVERLAP_CHARS = int(os.getenv("CHUNK_OVERLAP_CHARS", "400"))
MAX_CLAUSES = int(os.getenv("MAX_CLAUSES", "20"))
TOP_K_PRECEDENTS = int(os.getenv("TOP_K_PRECEDENTS", "5"))

print("Loaded config.")

Loaded config.


In [4]:
# 2) Create clients
def get_openai_client() -> OpenAI:
    return OpenAI(
        api_key=AZURE_OPENAI_API_KEY,
        base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
    )

openai_client = get_openai_client()

search_index_client = SearchIndexClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    credential=AzureKeyCredential(AZURE_SEARCH_KEY),
)

search_client = SearchClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    index_name=AZURE_SEARCH_INDEX_NAME,
    credential=AzureKeyCredential(AZURE_SEARCH_KEY),
)

blob_service_client = BlobServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING)

print("Client is created")

Client is created


In [5]:
# 3) Utility functions
def safe_json_loads(text: str) -> Any:
    text = text.strip()

    if text.startswith("```"):
        parts = text.split("```")
        if len(parts) >= 2:
            text = parts[1].strip()

    start_candidates = [i for i in [text.find("{"), text.find("[")] if i != -1]
    if start_candidates:
        start = min(start_candidates)
        end = max(text.rfind("}"), text.rfind("]"))
        if end > start:
            text = text[start:end + 1]

    return json.loads(text)

def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        pages.append({
            "page_number": i,
            "text": page.extract_text() or ""
        })
    return pages

def chunk_text(text: str, max_chars: int = MAX_CHUNK_CHARS, overlap_chars: int = CHUNK_OVERLAP_CHARS) -> List[str]:
    text = " ".join(text.split())
    if not text:
        return []
    
    chunks = []
    start = 0
    n = len(text)

    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= n:
            break
        start = max(0, end - overlap_chars)

    return chunks

def embed_text(text: str) -> List[float]:
    resp = openai_client.embeddings.create(
        model=AZURE_OPENAI_EMBEDDING_MODEL,
        input=text[:12000],
    )
    return resp.data[0].embedding

def llm_chat(system_prompt: str, user_prompt: str, temperature: float = 0.0, max_tokens: int = 2000) -> str:
    resp = openai_client.chat.completions.create(
        model=AZURE_OPENAI_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content or ""

print("Utilities ready")

Utilities ready


In [6]:
# 4) Blob Storage upload
def ensure_container(container_name: str = AZURE_STORAGE_CONTAINER) -> None:
    container_client = blob_service_client.get_container_client(container_name)
    try:
        container_client.create_container()
    except Exception:
        pass

def upload_file_to_blob(file_path: str, container_name: str = AZURE_STORAGE_CONTAINER) -> str:
    ensure_container(container_name)
    file_path = Path(file_path)
    container_client = blob_service_client.get_container_client(container_name)
    blob_name = f"{file_path.stem}/{file_path.name}"
    blob_client = container_client.get_blob_client(blob_name)

    with open(file_path, "rb") as f:
        blob_client.upload_blob(f, overwrite=True)

    return blob_client.url

print("Blob functions ready")

Blob functions ready


In [7]:
# 5) Azure AI Search index
def ensure_search_index(vector_dimensions: int = 1536) -> None:
    try:
        search_index_client.get_index(AZURE_SEARCH_INDEX_NAME)
        print(f"Index already exists: {AZURE_SEARCH_INDEX_NAME}")
        return
    except Exception:
        pass

    index = SearchIndex(
        name=AZURE_SEARCH_INDEX_NAME,
        fields=[
            SimpleField(name="id", type=SearchFieldDataType.String, key=True, filterable=True),
            SimpleField(name="doc_type", type=SearchFieldDataType.String, filterable=True),
            SimpleField(name="document_name", type=SearchFieldDataType.String, filterable=True),
            SimpleField(name="source_blob_url", type=SearchFieldDataType.String, filterable=False),
            SimpleField(name="page_number", type=SearchFieldDataType.Int32, filterable=True, facetable=True),
            SimpleField(name="chunk_id", type=SearchFieldDataType.Int32, filterable=True, facetable=True),
            SimpleField(name="clause_type", type=SearchFieldDataType.String, filterable=True, facetable=True),
            SearchableField(name="content", type=SearchFieldDataType.String),
            SearchField(
                name="content_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                searchable=True,
                vector_search_dimensions=vector_dimensions,
                vector_search_profile_name="legal-vector-profile",
            ),
        ],
        vector_search=VectorSearch(
            algorithms=[HnswAlgorithmConfiguration(name="legal-hnsw")],
            profiles=[VectorSearchProfile(name="legal-vector-profile", algorithm_configuration_name="legal-hnsw")],
        ),
    )

    search_index_client.create_index(index)
    print(f"Created index: {AZURE_SEARCH_INDEX_NAME}")

def upsert_chunks(docs: List[Dict[str, Any]]) -> None:
    if docs:
        search_client.upload_documents(documents=docs)
        print(f"Uploaded {len(docs)} chunks to Azure AI Search.")

print("Search index helpers ready")

Search index helpers ready


In [8]:
# 6) Document ingestion agent
# This agent reads PDF pages, chunks the text, creates embeddings, uploads the chunks to Azure AI Search & stores the original PDF in Blob Storage
class IngestionAgent:
    def ingest_pdf(self, pdf_path: str, doc_type: str = "precedent") -> Dict[str, Any]:
        pdf_path_obj = Path(pdf_path)
        blob_url = upload_file_to_blob(str(pdf_path_obj))
        pages = extract_pdf_pages(str(pdf_path_obj))

        docs = []
        chunk_id = 0

        for page in pages:
            page_number = page["page_number"]
            page_text = page["text"]
            for chunk in chunk_text(page_text):
                docs.append({
                    "id": f"{pdf_path_obj.stem}-{page_number}-{chunk_id}",
                    "doc_type": doc_type,
                    "document_name": pdf_path_obj.name,
                    "source_blob_url": blob_url,
                    "page_number": page_number,
                    "chunk_id": chunk_id,
                    "clause_type": "raw_text",
                    "content": chunk,
                    "content_vector": embed_text(chunk),
                })
                chunk_id += 1

        upsert_chunks(docs)

        return {
            "pdf_path": str(pdf_path_obj),
            "blob_url": blob_url,
            "chunks_indexed": len(docs),
            "doc_type": doc_type,
        }

ingestion_agent = IngestionAgent()
print("Ingestion agent ready.")

Ingestion agent ready.


In [9]:
# 7) Clause extraction agent
class ClauseExtractionAgent:
    SYSTEM_PROMPT = """
You are a legal document clause extraction agent.

Extract only material clauses from the document.
Focus on sections like confidentiality, liability, termination, indemnity, governing law, payment, compliance, and data protection.

Return STRICT JSON only in this schema:

{
  "clauses": [
    {
      "clause_id": "C1",
      "title": "Clause title",
      "clause_type": "confidentiality|termination|liability|payment|governing_law|compliance|data_protection|indemnity|other",
      "risk_level": "low|medium|high",
      "text": "full clause text",
      "obligations": ["..."],
      "exceptions": ["..."],
      "notes": "short note"
    }
  ]
}
"""

    def extract_clauses(self, document_text: str) -> List[Dict[str, Any]]:
        user_prompt = f"Document text:\n\n{document_text[:30000]}"
        raw = llm_chat(self.SYSTEM_PROMPT, user_prompt, temperature=0.0, max_tokens=4000)
        data = safe_json_loads(raw)
        return data.get("clauses", [])

clause_agent = ClauseExtractionAgent()
print("Clause extraction agent loaded")

Clause extraction agent loaded


In [10]:
# 8) Compliance validation agent
# For each extracted clause:
# retrieve similar precedent chunks from Azure AI Search, ask the model to classify the risk & return grounded findings with evidence
def hybrid_search(query: str, top_k: int = TOP_K_PRECEDENTS, doc_type: str = "precedent") -> List[Dict[str, Any]]:
    query_vector = embed_text(query)
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=top_k,
        fields="content_vector",
    )

    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        filter=f"doc_type eq '{doc_type}'",
        top=top_k,
        select=[
            "id",
            "doc_type",
            "document_name",
            "source_blob_url",
            "page_number",
            "chunk_id",
            "clause_type",
            "content",
        ],
    )

    return [dict(r) for r in results]

class ComplianceValidationAgent:
    SYSTEM_PROMPT = """
You are a legal compliance validation agent.

Given ONE clause and retrieved precedent snippets, assess whether the clause is:
- compliant
- non-compliant
- needs-review

Use only the provided evidence. Do not invent precedents.

Return STRICT JSON only:

{
  "clause_id": "C1",
  "status": "compliant|non-compliant|needs-review",
  "risk_level": "low|medium|high",
  "issue": "short issue description",
  "recommendation": "what to change",
  "evidence": [
    {
      "document_name": "file.pdf",
      "page_number": 1,
      "excerpt": "relevant snippet"
    }
  ]
}
"""

    def review_clause(self, clause: Dict[str, Any], precedents: List[Dict[str, Any]]) -> Dict[str, Any]:
        precedent_text = "\n\n".join(
            [
                f"- {p.get('document_name')} | page {p.get('page_number')} | {p.get('content')}"
                for p in precedents
            ]
        )

        user_prompt = f"""
Clause JSON:
{json.dumps(clause, indent=2)}

Retrieved precedents:
{precedent_text}
"""
        raw = llm_chat(self.SYSTEM_PROMPT, user_prompt, temperature=0.0, max_tokens=2500)
        return safe_json_loads(raw)

validation_agent = ComplianceValidationAgent()
print("Validation agent ready")

Validation agent ready


In [11]:
# 9) Summary agent
class SummaryAgent:
    SYSTEM_PROMPT = """
You are a grounded legal summary agent.

Summarize the review findings in a concise executive-friendly way.

Return STRICT JSON only:

{
  "overall_risk": "low|medium|high",
  "executive_summary": "2-4 sentences",
  "top_issues": ["..."],
  "recommended_actions": ["..."]
}
"""

    def summarize(self, document_name: str, findings: List[Dict[str, Any]]) -> Dict[str, Any]:
        user_prompt = f"""
Document: {document_name}

Findings:
{json.dumps(findings, indent=2)}
"""
        raw = llm_chat(self.SYSTEM_PROMPT, user_prompt, temperature=0.0, max_tokens=2000)
        return safe_json_loads(raw)

summary_agent = SummaryAgent()
print("Summary agent ready")

Summary agent ready


In [12]:
# 10) Orchestrator Agent
# Main workflow:
# 1. Ingest precedent documents
# 2. Read target document
# 3. Extract clauses
# 4. Retrieve similar precedent chunks
# 5. Validate each clause
# 6. Summarize results
class LegalDocumentReviewOrchestrator:
    def __init__(self):
        self.ingestion_agent = ingestion_agent
        self.clause_agent = clause_agent
        self.validation_agent = validation_agent
        self.summary_agent = summary_agent

    def ingest_folder(self, folder: str, doc_type: str = "precedent") -> List[Dict[str, Any]]:
        folder_path = Path(folder)
        outputs = []
        for pdf_file in folder_path.rglob("*.pdf"):
            outputs.append(self.ingestion_agent.ingest_pdf(str(pdf_file), doc_type=doc_type))
        return outputs

    def review_document(self, pdf_path: str) -> Dict[str, Any]:
        pdf_path_obj = Path(pdf_path)
        pages = extract_pdf_pages(str(pdf_path_obj))
        full_text = "\n\n".join([p["text"] for p in pages if p["text"].strip()])

        # Optionally store target doc too
        self.ingestion_agent.ingest_pdf(str(pdf_path_obj), doc_type="target")

        clauses = self.clause_agent.extract_clauses(full_text)[:MAX_CLAUSES]

        findings = []
        for clause in clauses:
            clause_text = clause.get("text", "")
            precedents = hybrid_search(clause_text, top_k=TOP_K_PRECEDENTS, doc_type="precedent")
            finding = self.validation_agent.review_clause(clause, precedents)
            findings.append(finding)

        summary = self.summary_agent.summarize(pdf_path_obj.name, findings)

        return {
            "document_name": pdf_path_obj.name,
            "clauses_reviewed": len(clauses),
            "findings": findings,
            "summary": summary,
        }

orchestrator = LegalDocumentReviewOrchestrator()
print("Orchestrator ready.")

Orchestrator ready.


In [13]:
# 11) Initialize the index if not done already
# Run this once before ingestion
ensure_search_index(vector_dimensions=1536)

Index already exists: legal_doc_review


In [14]:
# 12) Ingest precedent documents
precedent_results = orchestrator.ingest_folder("./data/precedents", doc_type="precedent")
precedent_results

[]

In [15]:
# 13) Review one target document
# Load a contract or policy PDF
report = orchestrator.review_document("C:\\Users\\agenticaiuser\\Downloads\\sample_legal_doc.pdf")
print(json.dumps(report, indent=2, ensure_ascii=False))

Uploaded 5 chunks to Azure AI Search.
{
  "document_name": "sample_legal_doc.pdf",
  "clauses_reviewed": 7,
  "findings": [
    {
      "clause_id": "C1",
      "status": "needs-review",
      "risk_level": "high",
      "issue": "The clause allows unrestricted internal disclosure of confidential information and removes liability for indirect consequences, which may conflict with standard confidentiality practices.",
      "recommendation": "Restrict internal disclosures to a 'need-to-know' basis and include liability for indirect consequences if caused by negligence or breach of confidentiality.",
      "evidence": []
    },
    {
      "clause_id": "C2",
      "status": "non-compliant",
      "risk_level": "high",
      "issue": "The clause imposes unlimited liability on the Client while completely exempting the Service Provider, which is unbalanced and may be deemed unconscionable or unenforceable in many jurisdictions.",
      "recommendation": "Revise the clause to include a mutua

In [16]:
# 14) Testing examples
sample_precedent_text = """
Confidentiality Clause:
The receiving party shall not disclose confidential information for a period of 5 years.

Liability Clause:
Total liability shall not exceed the contract value.

Governing Law:
This agreement is governed by Indian law.
"""

sample_bad_text = """
Confidentiality Clause:
The receiving party may disclose information internally without restriction.

Liability Clause:
The company has unlimited liability.

Termination Clause:
No termination rights are defined.
"""

print("Testing clause extraction on sample precedent text...")
clauses_good = clause_agent.extract_clauses(sample_precedent_text)
print(json.dumps(clauses_good, indent=2))

print("\nTesting clause extraction on risky sample text...")
clauses_bad = clause_agent.extract_clauses(sample_bad_text)
print(json.dumps(clauses_bad, indent=2))

Testing clause extraction on sample precedent text...
[
  {
    "clause_id": "C1",
    "title": "Confidentiality Clause",
    "clause_type": "confidentiality",
    "risk_level": "medium",
    "text": "The receiving party shall not disclose confidential information for a period of 5 years.",
    "obligations": [
      "The receiving party must maintain confidentiality for 5 years."
    ],
    "exceptions": [
      "Not specified in the clause."
    ],
    "notes": "Duration of confidentiality is explicitly defined."
  },
  {
    "clause_id": "C2",
    "title": "Liability Clause",
    "clause_type": "liability",
    "risk_level": "medium",
    "text": "Total liability shall not exceed the contract value.",
    "obligations": [
      "Liability is capped at the contract value."
    ],
    "exceptions": [
      "Not specified in the clause."
    ],
    "notes": "Liability cap is clear but may need further negotiation depending on the contract value."
  },
  {
    "clause_id": "C3",
    "ti

In [17]:
# 15) Manual validation test
mock_clause = {
    "clause_id": "C1",
    "title": "Confidentiality",
    "clause_type": "confidentiality",
    "risk_level": "high",
    "text": "The receiving party may disclose information internally without restriction.",
    "obligations": [],
    "exceptions": ["internal disclosure allowed"],
    "notes": "Broad disclosure right"
}

mock_precedents = [
    {
        "document_name": "precedent1.pdf",
        "page_number": 2,
        "content": "The receiving party shall not disclose confidential information except as required by law."
    },
    {
        "document_name": "precedent2.pdf",
        "page_number": 4,
        "content": "Confidential information must remain restricted to authorized personnel only."
    }
]

test_finding = validation_agent.review_clause(mock_clause, mock_precedents)
print(json.dumps(test_finding, indent=2, ensure_ascii=False))

{
  "clause_id": "C1",
  "status": "non-compliant",
  "risk_level": "high",
  "issue": "The clause allows unrestricted internal disclosure, which conflicts with standard confidentiality practices.",
  "recommendation": "Restrict internal disclosure to authorized personnel only and align with legal requirements.",
  "evidence": [
    {
      "document_name": "precedent1.pdf",
      "page_number": 2,
      "excerpt": "The receiving party shall not disclose confidential information except as required by law."
    },
    {
      "document_name": "precedent2.pdf",
      "page_number": 4,
      "excerpt": "Confidential information must remain restricted to authorized personnel only."
    }
  ]
}


In [18]:
# End of notebook